In [ ]:
import sys

sys.path.append("..")

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

from SpecEmbedding.const import gnps, mona
from SpecEmbedding.data.tokenizer import Tokenizer
from SpecEmbedding.trainer.trainer import ModelTester
from SpecEmbedding.utils.model import (
    cosine_similarity,
    embedding,
    load_transformer_model,
    search,
    search_with_spectra,
)

In [2]:
spectra_paths = {
    "gnps":{
        "orbitrap": {
            "train": (gnps.ORBITRAP_TRAIN_QUERY, gnps.ORBITRAP_TEST_REF),
            "test": (gnps.ORBITRAP_TEST_QUERY, gnps.ORBITRAP_TEST_REF)
        },
        "qtof": {
            "test": (gnps.QTOF_TEST_QUERY, gnps.QTOF_TEST_REF)
        },
        "other": {
            "test": (gnps.OTHER_TEST_QUERY, gnps.OTHER_TEST_REF)
        }
    }
}
gnps_train_ref = np.load(gnps.ORBITRAP_TRAIN_REF, allow_pickle=True)

In [3]:
show_progress_bar = True
is_augment = True
model_backbone = "transformer"
loss_type = "SupConWithTanimotoLoss"
replica_suffix = "-replication-{}"
k_metric = [5, 1, 10]
batch_size = None
loader_batch_size = 4096
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
tokenizer = Tokenizer(100, show_progress_bar)
model = load_transformer_model(device, loss_type, is_augment)

tester = ModelTester(model, device, show_progress_bar)

In [4]:
replica_df_seq = []

for i in tqdm(range(10)):
    df_seq = []
    for db, db_metadata in spectra_paths.items():
        for desc, path_metadata in db_metadata.items():
            for info, paths in path_metadata.items():
                print("-" * 40, f"{db}-{desc}-{info}", "-" * 40)
                query_path, ref_path = paths
                query_path = query_path.with_stem(query_path.stem + replica_suffix.format(i + 1))
                ref_path = ref_path.with_stem(ref_path.stem + replica_suffix.format(i + 1))
                if db == "gnps" and desc == "orbitrap":
                    if info == "train":
                        query_path = gnps.ORBITRAP_TRAIN_QUERY
                    
                    ref_spectra = np.load(ref_path, allow_pickle=True)
                    query_spectra = np.load(query_path, allow_pickle=True)
                    ref_spectra = np.hstack((gnps_train_ref, ref_spectra))
                    df = search_with_spectra(
                        f"{db}-{desc}-{info}", tester,
                        k_metric, tokenizer,
                        query_spectra, ref_spectra,
                        loader_batch_size,
                        show_progress_bar, batch_size
                    )
                else:
                    df = search(
                    f"{db}-{desc}-{info}", tester, 
                    k_metric, tokenizer,
                    query_path, ref_path, 
                    loader_batch_size,
                    show_progress_bar, 512
                )
                df_seq.append(df)
    df = pd.concat(df_seq, axis=0)
    print(df)
    replica_df_seq.append(df)

  0%|          | 0/10 [00:00<?, ?it/s]

---------------------------------------- gnps-orbitrap-train ----------------------------------------


calculate hit and recall count: 100%|██████████| 1/1 [00:09<00:00,  9.46s/it]


---------------------------------------- gnps-orbitrap-test ----------------------------------------


calculate hit and recall count: 100%|██████████| 1/1 [00:02<00:00,  2.37s/it]


...

---------------------------------------- gnps-qtof-test ----------------------------------------


calculate hit and recall count: 100%|██████████| 15/15 [00:07<00:00,  1.98it/s]


---------------------------------------- gnps-other-test ----------------------------------------


100%|██████████| 10/10 [24:34<00:00, 147.46s/it]

                         top1      top5     top10
gnps-orbitrap-train  0.804116  0.915341  0.938549
gnps-orbitrap-test   0.822064  0.923488  0.944247
gnps-qtof-test       0.501064  0.694149  0.750798
gnps-other-test      0.814757  0.942180  0.959541


In [5]:
data = []
indices = replica_df_seq[0].index
columns = replica_df_seq[0].columns
for item in replica_df_seq:
    data.append([item.values])

In [6]:
data = np.concatenate(data, axis=0)
np.set_printoptions(precision=5, suppress=True)
np.mean(data, axis=0) * 100, np.std(data, axis=0) * 100

(array([[80.46125, 91.5443 , 93.88264],
        [81.5777 , 92.54448, 94.78648],
        [50.31782, 69.37234, 75.01463],
        [81.0045 , 94.03968, 95.85646]]),
 array([[0.05351, 0.02532, 0.02481],
        [0.39894, 0.27699, 0.24946],
        [0.18857, 0.25194, 0.18829],
        [0.34954, 0.17293, 0.12383]]))

In [7]:
pd.set_option('display.precision', 4)
mean_df = pd.DataFrame(np.mean(data, axis=0) * 100, index=indices, columns=columns)
std_df = pd.DataFrame(np.std(data, axis=0) * 100, index=indices, columns=columns)

In [8]:
show_progress_bar = False
is_augment = True
model_backbone = "transformer"
loss_type = "SupConWithTanimotoLoss"
replica_suffix = "-replication-{}"
k_metric = [5, 1, 10]
batch_size = None
loader_batch_size = 512
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
tokenizer = Tokenizer(100, show_progress_bar)
model = load_transformer_model(device, loss_type, is_augment)
tester = ModelTester(model, device, show_progress_bar)

query_spectra = np.load("../data/legacy/MSBert/MTBLS1572/query.npy", allow_pickle=True)
ref_spectra = np.load("../data/legacy/MSBert/MTBLS1572/ref.npy", allow_pickle=True)

search_with_spectra(
    "MTBLS1572", tester,
    k_metric, tokenizer,
    query_spectra, ref_spectra,
    loader_batch_size,
    show_progress_bar, batch_size
)

,top1,top5,top10
MTBLS1572,1.0,1.0,1.0


In [9]:
query_embedding, _ = embedding(
    tester, tokenizer,
    512, query_spectra,
    False
)

ref_embedding, _ = embedding(
    tester, tokenizer,
    512, ref_spectra,
    False
)
cosine_score = cosine_similarity(
    query_embedding, ref_embedding
)
for i, j in enumerate(np.argmax(cosine_score, axis=1)):
    if i != j:
        print(f"{i}-th answer is [{ref_spectra[i].get("compound_name")}] but get [{ref_spectra[j].get("compound_name")}]")

In [10]:
show_progress_bar = True
is_augment = True
model_backbone = "transformer"
loss_type = "SupConWithTanimotoLoss"
k_metric = [5, 1, 10]
batch_size = None
loader_batch_size = 512
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
tokenizer = Tokenizer(100, show_progress_bar)
model = load_transformer_model(device, loss_type, is_augment)
tester = ModelTester(model, device, show_progress_bar)

In [11]:
query_spectra = np.load(mona.ORBITRAP_COMMON, allow_pickle=True)
ref_spectra = np.load(gnps.ORBITRAP_ALL, allow_pickle=True)

search_with_spectra(
    "Orbitrap Common", tester,
    k_metric, tokenizer,
    query_spectra, ref_spectra,
    loader_batch_size,
    show_progress_bar, batch_size
)

calculate hit and recall count: 100%|██████████| 1/1 [00:04<00:00,  4.01s/it]


,top1,top5,top10
Orbitrap Common,0.8525,0.9171,0.9396


In [12]:
query_spectra = np.load(mona.QTOF_COMMON, allow_pickle=True)
ref_spectra = np.load(gnps.QTOF_ALL, allow_pickle=True)

search_with_spectra(
    "QTOF Common", tester,
    k_metric, tokenizer,
    query_spectra, ref_spectra,
    loader_batch_size,
    show_progress_bar, batch_size
)

calculate hit and recall count: 100%|██████████| 1/1 [00:01<00:00,  1.85s/it]


,top1,top5,top10
QTOF Common,0.9845,0.9936,0.9977
